In [1]:
import pandas as pd

sg = pd.read_csv("../data/raw/wvs7_singapore.csv", sep=";", index_col=False)

sg.head()


,version,doi,A_YEAR,B_COUNTRY,B_COUNTRY_ALPHA,C_COW_NUM,C_COW_ALPHA,D_INTERVIEW,FW_START,FW_END,...,WEIGHT4A,WEIGHT4B,RESEMAVALBWGT,RESEMAVALWGT,SECVALBWGT,Y001_1,Y001_2,Y001_3,Y001_4,Y001_5
0,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070001.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,0.83,0.0,0.0,0.0,1.0,0.0
1,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070002.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,0.83,1.0,1.0,1.0,0.0,1.0
2,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070003.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,1.00,1.0,0.0,1.0,0.0,1.0
3,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070004.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,1.00,1.0,0.0,0.0,0.0,0.0
4,6-0-0 (2024-04-15),doi.org/10.14281/18241.20,2020.0,702.0,SGP,830.0,SIN,702070005.0,201911.0,202003.0,...,1.0,1.0,1.0,1.0,1.00,0.0,0.0,0.0,0.0,0.0


In [23]:
clean = sg[
    [
        "D_INTERVIEW",
        "Q275",
        "Q288",
        "Q279",
        "Q273",
        "Q144",
        "Q145"
    ]
].copy()

In [24]:
variables = ["Q275", "Q288", "Q279", "Q273", "Q144", "Q145"]

sg[variables].head()


,Q275,Q288,Q279,Q273,Q144,Q145
0,7.0,5.0,1.0,1.0,2.0,2.0
1,6.0,1.0,1.0,6.0,-2.0,1.0
2,6.0,4.0,1.0,6.0,2.0,2.0
3,6.0,6.0,1.0,1.0,2.0,2.0
4,3.0,8.0,5.0,1.0,2.0,2.0


In [25]:
for var in variables:
    print(f"\n--- {var} ---")
    print(sg[var].value_counts(dropna=False).sort_index())



--- Q275 ---
Q275
-2.0      4
 0.0    117
 1.0    126
 2.0    130
 3.0    341
 4.0    272
 5.0    348
 6.0    502
 7.0    155
 8.0     17
Name: count, dtype: int64

--- Q288 ---
Q288
-2.0      56
 1.0      79
 2.0      77
 3.0     228
 4.0     239
 5.0     588
 6.0     405
 7.0     250
 8.0      74
 9.0       6
 10.0     10
Name: count, dtype: int64

--- Q279 ---
Q279
1.0    971
2.0    186
3.0    135
4.0    271
5.0    216
6.0     89
7.0    128
8.0     16
Name: count, dtype: int64

--- Q273 ---
Q273
1.0    1188
2.0      21
3.0      88
4.0      18
5.0      96
6.0     601
Name: count, dtype: int64

--- Q144 ---
Q144
-2.0       1
 1.0      79
 2.0    1932
Name: count, dtype: int64

--- Q145 ---
Q145
-5.0      10
-2.0       3
-1.0       6
 1.0      66
 2.0    1927
Name: count, dtype: int64


## Q279: Employment status

Q279 is recoded into a binary employment variable.

- 1 = Employed (Q279 = 1, 2, 3)
- 0 = Unemployed (Q279 = 4, 5, 6, 7, 8)


In [26]:
employment_map = {
    1: 1,
    2: 1,
    3: 1,
    4: 0,
    5: 0,
    6: 0,
    7: 0,
    8: 0
}

sg["employment"] = sg["Q279"].map(employment_map)



In [27]:
pd.crosstab(sg["Q279"], sg["employment"], dropna=False)


employment,0,1
Q279,,
1.0,0,971
2.0,0,186
3.0,0,135
4.0,271,0
5.0,216,0
6.0,89,0
7.0,128,0
8.0,16,0


In [28]:
sg["employment"].value_counts(dropna=False)

employment
1    1292
0     720
Name: count, dtype: int64

## Q273: Marital status

Q273 is recoded into a binary marital status variable.

- 1 = Married (Q273 = 1, 2)
- 0 = Unmarried (Q273 = 3, 4, 5, 6)

In [29]:
marital_map = {
    1: 1,
    2: 1,
    3: 0,
    4: 0,
    5: 0,
    6: 0
}
sg["married"] = sg["Q273"].map(marital_map)

pd.crosstab(sg["Q273"], sg["married"], dropna=False)

married,0,1
Q273,,
1.0,0,1188
2.0,0,21
3.0,88,0
4.0,18,0
5.0,96,0
6.0,601,0


In [30]:
sg["married"].value_counts(dropna=False)

married
1    1209
0     803
Name: count, dtype: int64

Q275 measures the respondent's highest educational level using the ISCED-2011 classification, ranging from 0 (early childhood/no education) to 8 (doctoral or equivalent).

The missing-value code -2 ("No answer/refused") is recoded as missing (NaN).
Higher values indicate higher levels of education.


In [31]:
import numpy as np

sg["education"] = sg["Q275"].replace({
    -1: np.nan,
    -2: np.nan,
    -3: np.nan,
    -5: np.nan
})

In [32]:
pd.crosstab(
    sg["Q275"],
    sg["education"],
    dropna=False
)



sg["education"].value_counts(dropna=False).sort_index()

education
0.0    117
1.0    126
2.0    130
3.0    341
4.0    272
5.0    348
6.0    502
7.0    155
8.0     17
NaN      4
Name: count, dtype: int64

## Q288: Household income

Q288 measures the household's income level on a 10-step income scale.

Valid responses range from 1 to 10. Higher values indicate a higher household income group.

The missing-value code -2 ("No answer/refused") is recoded as missing (NaN).


In [33]:
sg["household_income"] = sg["Q288"].replace({
    -1: np.nan,
    -2: np.nan,
    -3: np.nan,
    -5: np.nan
})


In [34]:
sg["household_income"].value_counts(dropna=False).sort_index()

household_income
1.0      79
2.0      77
3.0     228
4.0     239
5.0     588
6.0     405
7.0     250
8.0      74
9.0       6
10.0     10
NaN      56
Name: count, dtype: int64

In [35]:
pd.crosstab(
    sg["Q288"],
    sg["household_income"],
    dropna=False
)

household_income,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,NaN
Q288,,,,,,,,,,,
-2.0,0,0,0,0,0,0,0,0,0,0,56
1.0,79,0,0,0,0,0,0,0,0,0,0
2.0,0,77,0,0,0,0,0,0,0,0,0
3.0,0,0,228,0,0,0,0,0,0,0,0
4.0,0,0,0,239,0,0,0,0,0,0,0
5.0,0,0,0,0,588,0,0,0,0,0,0
6.0,0,0,0,0,0,405,0,0,0,0,0
7.0,0,0,0,0,0,0,250,0,0,0,0
8.0,0,0,0,0,0,0,0,74,0,0,0


## Q144–Q145: Personal or family crime victimisation

Q144 and Q145 are combined into a binary crime victimisation variable.

- 1 = Victimised: either the respondent or a family member answered Yes
- 0 = Not victimised: both answered No
- Missing responses are treated as missing unless the other question provides a valid Yes response.

In [36]:
crime_missing = [-1, -2, -3, -5]

q144_clean = sg["Q144"].replace(crime_missing, np.nan)
q145_clean = sg["Q145"].replace(crime_missing, np.nan)


In [37]:
sg["crime_victim"] = np.where(
    (q144_clean == 1) | (q145_clean == 1), 1,
    np.where(
        (q144_clean == 2) & (q145_clean == 2), 0, np.nan))

In [38]:
pd.crosstab(
    [sg["Q144"], sg["Q145"]],
    sg["crime_victim"],
    dropna=False)

crime_victim   0.0  1.0  NaN
Q144 Q145                   
-2.0 -5.0        0    0    0
     -2.0        0    0    0
     -1.0        0    0    0
      1.0        0    1    0
      2.0        0    0    0
 1.0 -5.0        0    1    0
     -2.0        0    0    0
     -1.0        0    2    0
      1.0        0   22    0
      2.0        0   54    0
 2.0 -5.0        0    0    9
     -2.0        0    0    3
     -1.0        0    0    4
      1.0        0   43    0
      2.0     1873    0    0

In [39]:
sg["crime_victim"].value_counts(dropna=False)


crime_victim
0.0    1873
1.0     123
NaN      16
Name: count, dtype: int64

# Descriptive Statistics

This section presents descriptive statistics for the cleaned Singapore variables. 
For continuous and ordinal variables, summary statistics including the mean, median, 
standard deviation, variance, minimum, and maximum are reported. For binary variables, 
frequency distributions and percentages are also examined.


In [40]:
analysis_vars = [
    "education",
    "household_income",
    "employment",
    "married",
    "crime_victim"
]

In [41]:
descriptive_stats = sg[analysis_vars].agg([
    "count",
    "mean",
    "median",
    "std",
    "var",
    "min",
    "max"
]).T

descriptive_stats

,count,mean,median,std,var,min,max
education,2008.0,4.218127,5.0,1.970695,3.883637,0.0,8.0
household_income,1956.0,4.979039,5.0,1.680966,2.825647,1.0,10.0
employment,2012.0,0.642147,1.0,0.479488,0.229908,0.0,1.0
married,2012.0,0.600895,1.0,0.489836,0.239940,0.0,1.0
crime_victim,1996.0,0.061623,0.0,0.240530,0.057855,0.0,1.0


In [42]:
binary_vars = ["employment", "married", "crime_victim"]

for var in binary_vars:
    print(f"\n--- {var} ---")
    print("Frequency:")
    print(sg[var].value_counts(dropna=False).sort_index())



--- employment ---
Frequency:
employment
0     720
1    1292
Name: count, dtype: int64

--- married ---
Frequency:
married
0     803
1    1209
Name: count, dtype: int64

--- crime_victim ---
Frequency:
crime_victim
0.0    1873
1.0     123
NaN      16
Name: count, dtype: int64


In [43]:
 print("\nPercentage:")
print(sg[var]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
        .mul(100)
        .round(2))



Percentage:
crime_victim
0.0    93.09
1.0     6.11
NaN     0.80
Name: proportion, dtype: float64


#Education has a mean of 4.22 (SD = 1.97), while household income has a mean of 4.98 (SD = 1.68). As both variables are measured using ordered categories in the WVS, these means represent average positions on their respective scales rather than actual years of education or monetary income.

Among the 2,012 respondents, 64.2% were classified as employed and 60.1% as married or living together as married. The "not employed" category should not be interpreted as equivalent to unemployment, as it also includes respondents such as retirees, students, and homemakers.

Personal or family crime victimisation was relatively uncommon. Among the 1,996 respondents with sufficient information, 123 (approximately 6.2%) were classified as having experienced personal or family crime victimisation.


In [44]:
print("Rows:", len(clean))
print("Unique IDs:", clean["D_INTERVIEW"].nunique())
print("Duplicate IDs:", clean["D_INTERVIEW"].duplicated().sum())

Rows: 2012
Unique IDs: 2012
Duplicate IDs: 0


In [45]:
clean.to_csv(
    "../data/processed/clean_sg_part_b_junhui.csv",
    index=False)